In [34]:
# ============================================================
# INSTALL
# ============================================================
!pip install transformers datasets accelerate torch -q

In [35]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
import torch
from torch import nn
from torch.utils.data import DataLoader
from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorWithPadding
)

device = torch.device("cuda:0")

MAX_LENGTH = 256
MAX_NEW_TOKEN = 50
BATCH_SIZE = 4
EPOCHS = 5
OUTPUT_SFT_MODEL_PATH = "out/sft_model"
OUTPUT_PPO_MODEL_PATH = "out/ppo_model"
OUTPUT_DPO_MODEL_PATH = "out/dpo_model"

os.makedirs(OUTPUT_SFT_MODEL_PATH, exist_ok=True)
os.makedirs(OUTPUT_PPO_MODEL_PATH, exist_ok=True)
os.makedirs(OUTPUT_DPO_MODEL_PATH, exist_ok=True)

In [36]:
def tokenize_pair(tokenizer, prompt, response):

    p_enc = tokenizer(prompt, truncation=True, max_length=MAX_LENGTH)

    full_enc = tokenizer(
        prompt + response,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors="pt"
    )

    input_ids = full_enc.input_ids.squeeze(0)
    attention_mask = full_enc.attention_mask.squeeze(0)

    prompt_len = len(p_enc["input_ids"])

    return input_ids, attention_mask, prompt_len

def compute_logprob(model, input_ids, attention_mask, prompt_length):
    """Compute log probs for the generated part only."""
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits

    # Shift so that tokens < n predict n
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = input_ids[:, 1:].contiguous()
    shift_mask = attention_mask[:, 1:].contiguous()

    # We only care about the generated tokens, not the prompt
    # prompt_length - 1 because of the shift
    gen_mask = shift_mask.copy()
    for i in range(gen_mask.size(0)):
        gen_mask[i, :prompt_length-1] = 0

    log_probs = torch.nn.functional.log_softmax(shift_logits, dim=-1)
    token_logps = log_probs.gather(-1, shift_labels.unsqueeze(-1)).squeeze(-1)

    return (token_logps * gen_mask).sum(dim=1)

def generate_response(model, tokenizer, prompts, attention_mask):

    with torch.no_grad():

        outputs = model.generate(
            input_ids=prompts,
            attention_mask=attention_mask,
            max_new_tokens=MAX_NEW_TOKEN,
            do_sample=True,
            top_p=0.8,
            temperature=0.5,
            pad_token_id=tokenizer.eos_token_id
        )

    # Compute attention_mask for generated sequences
    attention_mask = (outputs != tokenizer.pad_token_id).long()

    return outputs, attention_mask

In [37]:
dataset = load_dataset("HumanLLMs/Human-Like-DPO-Dataset", split="train")
dataset = dataset.train_test_split(test_size=0.2, seed=42)

train_pref = dataset["train"].select(range(2048))
val_pref = dataset["test"].select(range(512))

In [38]:
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left" # Set padding side to left for decoder-only models

In [39]:
sft_model = AutoModelForCausalLM.from_pretrained(model_name, device_map="cuda:0")
sft_model.resize_token_embeddings(len(tokenizer))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding(50257, 768)

In [40]:
def sft_tokenize(example):

    tokens = tokenizer(
        example["prompt"] + example["chosen"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

In [41]:
sft_train_dataset = train_pref.map(
    sft_tokenize,
    remove_columns=["prompt","chosen","rejected"]
)

sft_val_dataset = val_pref.map(
    sft_tokenize,
    remove_columns=["prompt","chosen","rejected"]
)

Map:   0%|          | 0/2048 [00:00<?, ? examples/s]

Map:   0%|          | 0/512 [00:00<?, ? examples/s]

In [42]:
collator = DataCollatorWithPadding(tokenizer)

sft_train_loader = DataLoader(
    sft_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator
)

sft_val_loader = DataLoader(
    sft_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator
)

In [43]:
optimizer = torch.optim.AdamW(sft_model.parameters(), lr=1e-5)

for epoch in range(EPOCHS):
    sft_model.train()
    train_total = 0

    for step, batch in enumerate(sft_train_loader):

        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = sft_model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_total += loss.item()

    train_loss = train_total / len(sft_train_loader)

    sft_model.eval()
    val_total = 0

    with torch.no_grad():
        for step, batch in enumerate(sft_val_loader):

            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = sft_model(**batch)
            loss = outputs.loss

            val_total += loss.item()

    val_loss = val_total / len(sft_val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

Epoch 1/5 - Train Loss: 2.5859, Val Loss: 1.8517
Epoch 2/5 - Train Loss: 1.8915, Val Loss: 1.7350
Epoch 3/5 - Train Loss: 1.7562, Val Loss: 1.6725
Epoch 4/5 - Train Loss: 1.6701, Val Loss: 1.6320
Epoch 5/5 - Train Loss: 1.6007, Val Loss: 1.6019


In [44]:
sft_model.save_pretrained(OUTPUT_SFT_MODEL_PATH)
tokenizer.save_pretrained(OUTPUT_SFT_MODEL_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('out/sft_model/tokenizer_config.json', 'out/sft_model/tokenizer.json')

# PPO

In [45]:
class RewardModel(nn.Module):

    def __init__(self, base):
        super().__init__()

        self.base = base
        self.head = nn.Linear(base.config.n_embd,1)

    def forward(self,input_ids,attention_mask):

        out = self.base(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )

        hidden = out.hidden_states[-1]

        pooled = (hidden * attention_mask.unsqueeze(-1)).sum(1) / attention_mask.sum(1, keepdim=True)

        return self.head(pooled).squeeze(-1)

In [46]:
def prepare_reward(example):

    c_ids,c_mask,_ = tokenize_pair(tokenizer,example["prompt"],example["chosen"])
    r_ids,r_mask,_ = tokenize_pair(tokenizer,example["prompt"],example["rejected"])

    return {
        "chosen_ids":c_ids,
        "chosen_mask":c_mask,
        "rejected_ids":r_ids,
        "rejected_mask":r_mask
    }

In [47]:
reward_train_dataset = train_pref.map(
    prepare_reward,
    remove_columns=["prompt","chosen","rejected"]
)

reward_val_dataset = val_pref.map(
    prepare_reward,
    remove_columns=["prompt","chosen","rejected"]
)

Map:   0%|          | 0/2048 [00:00<?, ? examples/s]

Map:   0%|          | 0/512 [00:00<?, ? examples/s]

In [48]:
class RewardCollator:

    def __call__(self, batch):

        return {

            "chosen_ids": torch.stack(
                [torch.tensor(x["chosen_ids"]) for x in batch]
            ),

            "chosen_mask": torch.stack(
                [torch.tensor(x["chosen_mask"]) for x in batch]
            ),

            "rejected_ids": torch.stack(
                [torch.tensor(x["rejected_ids"]) for x in batch]
            ),

            "rejected_mask": torch.stack(
                [torch.tensor(x["rejected_mask"]) for x in batch]
            ),
        }

In [49]:
reward_train_loader = DataLoader(
    reward_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=RewardCollator()
)

reward_val_loader = DataLoader(
    reward_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=RewardCollator()
)

In [50]:
base = AutoModelForCausalLM.from_pretrained(OUTPUT_SFT_MODEL_PATH, device_map="cuda:0")
base.resize_token_embeddings(len(tokenizer))

reward_model = RewardModel(base).to(device)

optimizer = torch.optim.AdamW(reward_model.parameters(), lr=1e-6)

loss_fn = nn.BCEWithLogitsLoss()

best_val = float("inf")

for epoch in range(EPOCHS):
    reward_model.train()
    train_total = 0

    for step, batch in enumerate(reward_train_loader):
        c_ids = batch["chosen_ids"].to(device)
        c_mask = batch["chosen_mask"].to(device)
        r_ids = batch["rejected_ids"].to(device)
        r_mask = batch["rejected_mask"].to(device)

        c = reward_model(c_ids, c_mask)
        r = reward_model(r_ids, r_mask)
        loss = loss_fn(c - r, torch.ones_like(c))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_total += loss.item()

        if step % 50 == 0:
            print(f"[Step {step}] Loss: {loss.item():.4f}")

    val_total = 0
    reward_model.eval()
    with torch.no_grad():
        for batch in reward_val_loader:
            c_ids, c_mask = batch["chosen_ids"].to(device), batch["chosen_mask"].to(device)
            r_ids, r_mask = batch["rejected_ids"].to(device), batch["rejected_mask"].to(device)
            c, r = reward_model(c_ids, c_mask), reward_model(r_ids, r_mask)
            val_total += loss_fn(c - r, torch.ones_like(c)).item()

    val_loss = val_total / len(reward_val_loader)
    print(f"Epoch {epoch+1} Val Loss: {val_loss:.4f}")
    if val_loss < best_val:
        best_val = val_loss
        torch.save(reward_model.state_dict(), "best_reward_model.pt")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[Step 0] Loss: 0.7702
[Step 50] Loss: 0.5211
[Step 100] Loss: 0.5194
[Step 150] Loss: 0.3913
[Step 200] Loss: 0.5078
[Step 250] Loss: 0.2221
[Step 300] Loss: 0.4684
[Step 350] Loss: 0.1989
[Step 400] Loss: 0.2088
[Step 450] Loss: 0.0866
[Step 500] Loss: 0.0678
Epoch 1 Val Loss: 0.0691
[Step 0] Loss: 0.0179
[Step 50] Loss: 0.0624
[Step 100] Loss: 0.0302
[Step 150] Loss: 0.0070
[Step 200] Loss: 0.0043
[Step 250] Loss: 0.0102
[Step 300] Loss: 0.0606
[Step 350] Loss: 0.0044
[Step 400] Loss: 0.0110
[Step 450] Loss: 0.0019
[Step 500] Loss: 0.0087
Epoch 2 Val Loss: 0.0040
[Step 0] Loss: 0.0011
[Step 50] Loss: 0.0027
[Step 100] Loss: 0.0007
[Step 150] Loss: 0.0019
[Step 200] Loss: 0.0006
[Step 250] Loss: 0.0002
[Step 300] Loss: 0.0011
[Step 350] Loss: 0.0031
[Step 400] Loss: 0.0002
[Step 450] Loss: 0.0004
[Step 500] Loss: 0.0002
Epoch 3 Val Loss: 0.0011
[Step 0] Loss: 0.0001
[Step 50] Loss: 0.0003
[Step 100] Loss: 0.0001
[Step 150] Loss: 0.0006
[Step 200] Loss: 0.0001
[Step 250] Loss: 0.0010
[

In [51]:
ppo_policy = AutoModelForCausalLM.from_pretrained(OUTPUT_SFT_MODEL_PATH, device_map="cuda:0")
ppo_policy.resize_token_embeddings(len(tokenizer))

ppo_ref = AutoModelForCausalLM.from_pretrained(OUTPUT_SFT_MODEL_PATH, device_map="cuda:0")
ppo_ref.resize_token_embeddings(len(tokenizer))

for p in ppo_ref.parameters():
    p.requires_grad=False

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [52]:
def prepare_ppo(example):

    tokens = tokenizer(
        example["prompt"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors="pt"
    )

    return {
        "input_ids":tokens.input_ids.squeeze(0),
        "attention_mask":tokens.attention_mask.squeeze(0)
    }

In [53]:
ppo_train_dataset = train_pref.map(
    prepare_ppo,
    remove_columns=["prompt","chosen","rejected"]
)

ppo_val_dataset = val_pref.map(
    prepare_ppo,
    remove_columns=["prompt","chosen","rejected"]
)

Map:   0%|          | 0/2048 [00:00<?, ? examples/s]

Map:   0%|          | 0/512 [00:00<?, ? examples/s]

In [54]:
class PPOCollator:

    def __call__(self,batch):

        return {

            "input_ids":torch.stack([torch.tensor(x["input_ids"]) for x in batch]),
            "attention_mask":torch.stack([torch.tensor(x["attention_mask"]) for x in batch])
        }

In [55]:
ppo_train_loader = DataLoader(
    ppo_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=PPOCollator()
)

ppo_val_loader = DataLoader(
    ppo_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=PPOCollator()
)

In [56]:
def compute_logprob(model, input_ids, attention_mask, prompt_length):
    """Compute log probs for the generated part only."""
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits

    # Shift so that tokens < n predict n
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = input_ids[:, 1:].contiguous()
    shift_mask = attention_mask[:, 1:].contiguous()

    # We only care about the generated tokens, not the prompt
    # prompt_length - 1 because of the shift
    gen_mask = shift_mask.clone() # Changed .copy() to .clone()
    for i in range(gen_mask.size(0)):
        gen_mask[i, :prompt_length-1] = 0

    log_probs = torch.nn.functional.log_softmax(shift_logits, dim=-1)
    token_logps = log_probs.gather(-1, shift_labels.unsqueeze(-1)).squeeze(-1)

    return (token_logps * gen_mask).sum(dim=1)

optimizer = torch.optim.AdamW(ppo_policy.parameters(), lr=1e-6)

beta = 0.02
clip_eps = 0.2

# Lists to store metrics for plotting
ppo_train_losses = []
ppo_train_rewards = []
ppo_train_kls = []

for epoch in range(EPOCHS):
    ppo_policy.train()
    total_loss, total_reward, total_kl = 0, 0, 0

    for step, batch in enumerate(ppo_train_loader):
        prompts = batch["input_ids"].to(device)
        prompt_mask = batch["attention_mask"].to(device)
        prompt_len = prompts.size(1)

        # 1. Generate
        generated, gen_mask = generate_response(ppo_policy, tokenizer, prompts, prompt_mask)
        generated, gen_mask = generated.to(device), gen_mask.to(device)

        # 2. Score
        reward = reward_model(generated, gen_mask)

        # 3. Policy vs Ref LogProbs (Prompt is ignored inside compute_logprob)
        logp = compute_logprob(ppo_policy, generated, gen_mask, prompt_len)
        logp_ref = compute_logprob(ppo_ref, generated, gen_mask, prompt_len)

        # 4. PPO Logic
        ratio = torch.exp(logp - logp_ref)
        clipped = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps)

        # KL is usually defined as log(p/ref)
        kl = (logp - logp_ref)
        policy_loss = -torch.min(ratio * reward, clipped * reward)

        loss = policy_loss.mean() + beta * kl.mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_reward += reward.mean().item()
        total_kl += kl.mean().item()

        ppo_train_losses.append(loss.item())
        ppo_train_rewards.append(reward.mean().item())
        ppo_train_kls.append(kl.mean().item())

        if step % 10 == 0:
            print(f"Step {step}: Loss {loss.item():.4f} | Reward {reward.mean().item():.4f} | KL {kl.mean().item():.4f}")

Step 0: Loss -0.6710 | Reward 1.4924 | KL -39.5360


OutOfMemoryError: CUDA out of memory. Tried to allocate 234.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 59.81 MiB is free. Including non-PyTorch memory, this process has 14.50 GiB memory in use. Of the allocated memory 14.15 GiB is allocated by PyTorch, and 222.70 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
test_prompt = "What is the meaning of life?"
inputs = tokenizer(test_prompt, return_tensors="pt").to(device)

# Generate from PPO Policy
ppo_policy.eval()
ppo_output = generate_response(ppo_policy, tokenizer, inputs.input_ids, inputs.attention_mask)[0]
ppo_text = tokenizer.decode(ppo_output[0], skip_special_tokens=True)

# Generate from SFT Model for comparison
sft_model.eval()
sft_output = generate_response(sft_model, tokenizer, inputs.input_ids, inputs.attention_mask)[0]
sft_text = tokenizer.decode(sft_output[0], skip_special_tokens=True)

print(f"Prompt: {test_prompt}\n")
print(f"SFT Model Response:\n{sft_text[len(test_prompt):].strip()}\n")
print(f"PPO Policy Response:\n{ppo_text[len(test_prompt):].strip()}")

# DPO

In [ ]:
import torch.nn.functional as F

# Initialize DPO models
dpo_policy = AutoModelForCausalLM.from_pretrained(OUTPUT_SFT_MODEL_PATH, device_map="cuda:0")
dpo_ref = AutoModelForCausalLM.from_pretrained(OUTPUT_SFT_MODEL_PATH, device_map="cuda:0")

for p in dpo_ref.parameters():
    p.requires_grad = False

def get_logps(logits, labels, attention_mask):
    labels = labels[:, 1:].clone()
    logits = logits[:, :-1, :]
    loss_mask = attention_mask[:, 1:].clone()

    log_probs = F.log_softmax(logits, dim=-1)
    per_token_logps = torch.gather(log_probs, dim=2, index=labels.unsqueeze(2)).squeeze(2)
    return (per_token_logps * loss_mask).sum(-1)

def dpo_loss(policy_chosen_logps, policy_rejected_logps, ref_chosen_logps, ref_rejected_logps, beta=0.1):
    pi_logratios = policy_chosen_logps - policy_rejected_logps
    ref_logratios = ref_chosen_logps - ref_rejected_logps

    logits = pi_logratios - ref_logratios
    losses = -F.logsigmoid(beta * logits)

    return losses.mean(), pi_logratios.detach().mean(), ref_logratios.detach().mean()

In [ ]:
def prepare_dpo_batch(batch):
    # Re-using tokenize_pair from earlier cells
    c_ids, c_mask, c_len = [], [], []
    r_ids, r_mask, r_len = [], [], []

    for item in batch:
        ci, cm, cl = tokenize_pair(tokenizer, item['prompt'], item['chosen'])
        ri, rm, rl = tokenize_pair(tokenizer, item['prompt'], item['rejected'])
        c_ids.append(ci); c_mask.append(cm); c_len.append(cl)
        r_ids.append(ri); r_mask.append(rm); r_len.append(rl)

    return {
        'chosen_input_ids': torch.stack(c_ids).to(device),
        'chosen_attention_mask': torch.stack(c_mask).to(device),
        'rejected_input_ids': torch.stack(r_ids).to(device),
        'rejected_attention_mask': torch.stack(r_mask).to(device)
    }

dpo_train_loader = DataLoader(train_pref, batch_size=BATCH_SIZE, shuffle=True, collate_fn=prepare_dpo_batch)
dpo_val_loader = DataLoader(val_pref, batch_size=BATCH_SIZE, shuffle=False, collate_fn=prepare_dpo_batch)

In [ ]:
optimizer = torch.optim.AdamW(dpo_policy.parameters(), lr=5e-7)
beta_dpo = 0.1

all_train_losses = []
all_pi_ratios = []
all_ref_ratios = []

for epoch in range(EPOCHS):
    dpo_policy.train()
    epoch_train_losses = []
    epoch_pi_ratios = []
    epoch_ref_ratios = []

    for step, batch in enumerate(dpo_train_loader):
        # Policy forward
        p_chosen_logits = dpo_policy(batch['chosen_input_ids'], attention_mask=batch['chosen_attention_mask']).logits
        p_rejected_logits = dpo_policy(batch['rejected_input_ids'], attention_mask=batch['rejected_attention_mask']).logits

        # Ref forward
        with torch.no_grad():
            r_chosen_logits = dpo_ref(batch['chosen_input_ids'], attention_mask=batch['chosen_attention_mask']).logits
            r_rejected_logits = dpo_ref(batch['rejected_input_ids'], attention_mask=batch['rejected_attention_mask']).logits

        p_chosen_logps = get_logps(p_chosen_logits, batch['chosen_input_ids'], batch['chosen_attention_mask'])
        p_rejected_logps = get_logps(p_rejected_logits, batch['rejected_input_ids'], batch['rejected_attention_mask'])
        r_chosen_logps = get_logps(r_chosen_logits, batch['chosen_input_ids'], batch['chosen_attention_mask'])
        r_rejected_logps = get_logps(r_rejected_logits, batch['rejected_input_ids'], batch['rejected_attention_mask'])

        loss, pi_ratio, ref_ratio = dpo_loss(p_chosen_logps, p_rejected_logps, r_chosen_logps, r_rejected_logps, beta=beta_dpo)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_train_losses.append(loss.item())
        epoch_pi_ratios.append(pi_ratio.item())
        epoch_ref_ratios.append(ref_ratio.item())

        if step % 50 == 0:
            print(f"Epoch {epoch+1}/{EPOCHS} - Step {step}: Loss {loss.item():.4f} | Pi Ratio {pi_ratio.item():.4f} | Ref Ratio {ref_ratio.item():.4f}")

    all_train_losses.extend(epoch_train_losses)
    all_pi_ratios.extend(epoch_pi_ratios)
    all_ref_ratios.extend(epoch_ref_ratios)

dpo_policy.save_pretrained(OUTPUT_DPO_MODEL_PATH)

In [ ]:
test_prompt = "What is the meaning of life?"
inputs = tokenizer(test_prompt, return_tensors="pt").to(device)

# Generate from DPO Policy
dpo_policy.eval()
dpo_output = generate_response(dpo_policy, tokenizer, inputs.input_ids, inputs.attention_mask)[0]
dpo_text = tokenizer.decode(dpo_output[0], skip_special_tokens=True)

# Generate from SFT Model for comparison
sft_model.eval()
sft_output = generate_response(sft_model, tokenizer, inputs.input_ids, inputs.attention_mask)[0]
sft_text = tokenizer.decode(sft_output[0], skip_special_tokens=True)

print(f"Prompt: {test_prompt}\n")
print(f"SFT Model Response:\n{sft_text[len(test_prompt):].strip()}\n")
print(f"DPO Policy Response:\n{dpo_text[len(test_prompt):].strip()}")

In [ ]:
import matplotlib.pyplot as plt

# Plotting the training loss
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.plot(ppo_train_losses)
plt.title('PPO Training Loss')
plt.xlabel('Step')
plt.ylabel('Loss')

# Plotting the rewards
plt.subplot(1, 3, 2)
plt.plot(ppo_train_rewards)
plt.title('PPO Training Rewards')
plt.xlabel('Step')
plt.ylabel('Reward')

# Plotting the KL divergence
plt.subplot(1, 3, 3)
plt.plot(ppo_train_kls)
plt.title('PPO Training KL Divergence')
plt.xlabel('Step')
plt.ylabel('KL Divergence')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Plotting the training loss
plt.figure(figsize=(12, 5))
plt.subplot(1, 3, 1)
plt.plot(all_train_losses)
plt.title('DPO Training Loss')
plt.xlabel('Step')
plt.ylabel('Loss')

# Plotting the policy log-ratios
plt.subplot(1, 3, 2)
plt.plot(all_pi_ratios)
plt.title('Policy Log-Ratios (pi_logratios)')
plt.xlabel('Step')
plt.ylabel('Log-Ratio')

# Plotting the reference log-ratios
plt.subplot(1, 3, 3)
plt.plot(all_ref_ratios)
plt.title('Reference Log-Ratios (ref_logratios)')
plt.xlabel('Step')
plt.ylabel('Log-Ratio')

plt.tight_layout()
plt.show()